# XGBoost (referência da comparação)

**Estado:** o que está executado aqui é o trial 0 e um estudo curto de 5 trials, só para provar que o fluxo funciona.
O N de trials ainda não foi definido e o conjunto de teste não é usado neste notebook.
O estudo curto fica em `resultados/curto/`, fora do Git, para o estudo completo não retomar a partir dele.

Modelo, faixas e trial 0 seguem a Frente 3 de `planos/rafael.md`.
O número de árvores sai da parada antecipada na validação da dobra, com teto de 10.000 e paciência 10, e não conta entre os 8 hiperparâmetros.
O peso de classe fica desligado, porque o oversampling já está nas dobras.

In [1]:
import sys

sys.path.insert(0, "..")

import platform
import time

import numpy as np
import optuna
import xgboost as xgb

import comum

print(sys.executable)
print("Python", platform.python_version(), "|", "xgboost", xgb.__version__, "| optuna", optuna.__version__)

C:\dev\Faculdade\churn_project\.venv\Scripts\python.exe
Python 3.12.10 | xgboost 3.4.1 | optuna 5.0.0


In [2]:
MODELO = "xgboost"
N_TRIALS = 5  # parâmetro: o N final ainda não foi definido
ESTUDO_CURTO = True  # False grava em resultados/xgboost.db, que é o estudo que conta
N_JOBS = 2  # a máquina local tem pouca RAM livre
TETO_ARVORES = 10_000
PACIENCIA = 10

# O teste, que vem em `final`, fica de fora de propósito.
folds, _, colunas = comum.carregar_dobras()
print(len(folds), "dobras,", len(colunas), "colunas")

5 dobras, 28 colunas


In [3]:
PADRAO = dict(
    max_depth=6,
    min_child_weight=1.0,
    learning_rate=0.3,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_lambda=1.0,
    gamma=0.0,
    objective="binary:logistic",
)


def sugerir(t):
    return dict(
        max_depth=t.suggest_int("max_depth", 2, 10),
        min_child_weight=t.suggest_float("min_child_weight", 0.1, 20, log=True),
        learning_rate=t.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=t.suggest_float("subsample", 0.5, 1),
        colsample_bytree=t.suggest_float("colsample_bytree", 0.5, 1),
        reg_lambda=t.suggest_float("reg_lambda", 1e-3, 100, log=True),
        gamma=t.suggest_float("gamma", 0, 5),
        objective=t.suggest_categorical("objective", ["binary:logistic", "reg:squarederror"]),
    )


def ajustador(hp, arvores):
    """Devolve o `ajustar` de comum.avaliar_cv; anota em `arvores` quantas cada dobra usou."""

    def ajustar(dobra):
        Xt, yt, Xv, yv = dobra
        classificacao = hp["objective"] == "binary:logistic"
        classe = xgb.XGBClassifier if classificacao else xgb.XGBRegressor
        modelo = classe(
            **hp,
            n_estimators=TETO_ARVORES,
            early_stopping_rounds=PACIENCIA,
            eval_metric="logloss" if classificacao else "rmse",
            random_state=comum.SEMENTE,
            n_jobs=N_JOBS,
        )
        modelo.fit(Xt, yt, eval_set=[(Xv, yv)], verbose=False)
        arvores.append(modelo.best_iteration + 1)
        if classificacao:
            return lambda X: modelo.predict_proba(X)[:, 1]
        return modelo.predict  # pode sair de [0, 1]; ce e corrige_prior recortam

    return ajustar


def objetivo(trial):
    arvores = []
    r = comum.avaliar_cv(ajustador(sugerir(trial), arvores), folds, trial)
    trial.set_user_attr("arvores_dobras", arvores)
    return r["ks"]

## Trial 0

Configuração padrão do XGBoost, fora do estudo, para conferir contra a medida do plano.

In [4]:
arvores0 = []
t0 = time.perf_counter()
r0 = comum.avaliar_cv(ajustador(PADRAO, arvores0), folds)
print(f"KS médio {r0['ks']:.4f} | por dobra {np.round(r0['ks_dobras'], 4)}")
print(f"árvores por dobra {arvores0} | {time.perf_counter() - t0:.1f} s")
print(f"limiar fora da dobra {r0['limiar']:.4f}")
for parte in ("treino", "validacao"):
    nomes = ("ce", "mse", "ce_corr", "mse_corr")
    print(parte, {k: round(float(np.mean([d[parte][k] for d in r0["perdas_dobras"]])), 4) for k in nomes})
print("plano, taxa 0,3: KS 0,5207, 9 a 45 árvores, 1,2 s")

KS médio 0.5207 | por dobra [0.5174 0.5441 0.5081 0.4957 0.5381]
árvores por dobra [45, 22, 9, 18, 16] | 0.6 s
limiar fora da dobra 0.4350
treino {'ce': 0.3505, 'mse': 0.1113, 'ce_corr': 0.3066, 'mse_corr': 0.0977}
validacao {'ce': 0.4738, 'mse': 0.1582, 'ce_corr': 0.435, 'mse_corr': 0.1398}
plano, taxa 0,3: KS 0,5207, 9 a 45 árvores, 1,2 s


## Estudo

Com `n_startup_trials=5`, o `MedianPruner` não corta nada nos 5 primeiros trials, então o corte não é exercitado neste estudo curto.
Ele é coberto em `tests/test_comum.py`.

In [5]:
pasta = comum.PASTA_RESULTADOS / "curto" if ESTUDO_CURTO else comum.PASTA_RESULTADOS
pasta.mkdir(parents=True, exist_ok=True)
arquivo = pasta / f"{MODELO}.db"
if ESTUDO_CURTO:
    arquivo.unlink(missing_ok=True)
armazenamento = f"sqlite:///{arquivo.as_posix()}"

# Mesma configuração de estudo() em modelos_jeff.py, até o criar_estudo do Diego entrar no comum.py.
optuna.logging.set_verbosity(optuna.logging.WARNING)
estudo = optuna.create_study(
    study_name=MODELO,
    storage=armazenamento,
    load_if_exists=True,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=comum.SEMENTE),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
)
if not estudo.trials:
    estudo.enqueue_trial(PADRAO)  # trial 0 = padrão da biblioteca

restantes = N_TRIALS - sum(t.state.is_finished() for t in estudo.trials)
t0 = time.perf_counter()
if restantes > 0:
    estudo.optimize(objetivo, n_trials=restantes)
tempo_parede = estudo.user_attrs.get("tempo_parede_s", 0) + time.perf_counter() - t0
estudo.set_user_attr("tempo_parede_s", tempo_parede)
print(f"{restantes} trials em {tempo_parede:.1f} s")

5 trials em 10.8 s


In [6]:
linhas = estudo.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs"))
colunas_tabela = {"number": "trial", "value": "ks", "state": "estado", "user_attrs_tempo_s": "tempo_s"}
tabela = linhas.rename(columns=colunas_tabela)
tabela["arvores"] = tabela["user_attrs_arvores_dobras"]
tabela = tabela.drop(columns=[c for c in tabela if c.startswith("user_attrs_")])
tabela.columns = [c.removeprefix("params_") for c in tabela.columns]
print(tabela.round(4).T.to_string(header=False))

trial                               0                     1                     2                     3                          4
ks                             0.5207                0.5324                0.5207                0.5372                     0.5384
estado                       COMPLETE              COMPLETE              COMPLETE              COMPLETE                   COMPLETE
colsample_bytree                  1.0                 0.578                0.6062                0.6461                     0.5853
gamma                             0.0                0.2904                 0.917                2.2803                     4.7444
learning_rate                     0.3                0.1206                0.2708                0.0801                     0.0117
max_depth                           6                     5                     8                     5                          6
min_child_weight                  1.0               15.4036                0.1115  

In [7]:
relido = optuna.load_study(study_name=MODELO, storage=armazenamento)
assert [t.value for t in relido.trials] == [t.value for t in estudo.trials]  # o SQLite guardou o estudo
assert relido.trials[0].params == PADRAO
assert relido.trials[0].value == r0["ks"]  # o trial 0 do estudo repete a medida acima

melhor = estudo.best_trial
print("melhor trial:", melhor.number, "KS", round(melhor.value, 4))
print("KS por dobra:", np.round(melhor.user_attrs["ks_dobras"], 4))
print("árvores por dobra:", melhor.user_attrs["arvores_dobras"])
print("n_estimators do treino final:", round(np.mean(melhor.user_attrs["arvores_dobras"])))

melhor trial: 4 KS 0.5384
KS por dobra: [0.5617 0.5418 0.5284 0.5086 0.5513]
árvores por dobra: [410, 424, 267, 402, 408]
n_estimators do treino final: 382


In [8]:
resumo = comum.resumir_estudo(
    estudo,
    N_TRIALS,
    tempo_parede,
    param_perda="objective",
    perdas={"binary:logistic": "ce", "reg:squarederror": "mse"},
)
for campo, valor in resumo.items():
    print(campo, "=", valor)

n_trials = 5
n_completos = 5
n_podados = 0
por_perda = {'ce': {'n': 4, 'melhor': 0.5383649547429494, 'mediana': 0.5347721293088538, 'media_top5': 0.5321442067306432}, 'mse': {'n': 1, 'melhor': 0.5207128444069413, 'mediana': 0.5207128444069413, 'media_top5': 0.5207128444069413}}
melhor_por_categoria = {'objective': {'binary:logistic': 0.5383649547429494, 'reg:squarederror': 0.5207128444069413}}
importancias = {'learning_rate': 0.2750657059012304, 'gamma': 0.26603994665664066, 'min_child_weight': 0.15949388349992472, 'subsample': 0.13621549776574343, 'reg_lambda': 0.10099533040939655, 'colsample_bytree': 0.05268825935157206, 'max_depth': 0.009455189762237358, 'objective': 4.6186653254880434e-05}
config_trial0 = {'max_depth': 6, 'min_child_weight': 1.0, 'learning_rate': 0.3, 'subsample': 1.0, 'colsample_bytree': 1.0, 'reg_lambda': 1.0, 'gamma': 0.0, 'objective': 'binary:logistic'}
ks_trial0 = 0.5206676135619159
melhor_ks = 0.5383649547429494
convergencia = [0.5206676135619159, 0.532366626

## Próxima etapa

Ainda não executada, porque depende do N e usa o teste:

1. Estudo completo com `ESTUDO_CURTO = False` e o N definido.
2. Treino final por retreino: `final` do `carregar_dobras`, com `n_estimators` igual à média de `best_iteration + 1` nas cinco dobras do melhor trial e 5 sementes.
3. SHAP do modelo final, para a pergunta 6.
4. Gravação de `resultados/xgboost.json` com `comum.salvar_resultado`.